<a href="https://colab.research.google.com/github/ErickJLA/Co-Met/blob/main/cell2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ⚙️ 2. Data Ingestion 3.0
# =============================================================================
#  DATA INGESTION & COLUMN MAPPING
# =============================================================================
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np
import io
import gspread


# --- Configuration: Required Columns & Synonyms ---
RAW_COLUMN_SPECS = {
    'id':  ['id', 'study', 'study_id', 'paper', 'author'],
    'xe':  ['xe', 'mean_e', 'mean_exp', 'x_e', 'treatment_mean'],
    'sde': ['sde', 'sd_e', 'sd_exp', 'sigma_e'],
    'ne':  ['ne', 'n_e', 'n_exp', 'sample_e'],
    'xc':  ['xc', 'mean_c', 'mean_ctrl', 'x_c', 'control_mean'],
    'sdc': ['sdc', 'sd_c', 'sd_ctrl', 'sigma_c'],
    'nc':  ['nc', 'n_c', 'n_ctrl', 'sample_c']
}

BINARY_COLUMN_SPECS = {
    'id':          ['id', 'study', 'study_id', 'paper', 'author'],
    'events_e':    ['events_e', 'events_exp', 'cases_e', 'cases_exp', 'a', 'treatment_events'],
    'nonevents_e': ['nonevents_e', 'nonevents_exp', 'control_e', 'b', 'treatment_nonevents'],
    'events_c':    ['events_c', 'events_ctrl', 'cases_c', 'cases_ctrl', 'c', 'control_events'],
    'nonevents_c': ['nonevents_c', 'nonevents_ctrl', 'control_c', 'd', 'control_nonevents']
}

PRECALC_COLUMN_SPECS = {
    'id':       ['id', 'study', 'study_id', 'paper', 'author'],
    'yi':       ['yi', 'effect_size', 'es', 'hedges_g', 'lnrr', 'smd', 'effect', 'g', 'd'],
    'variance': ['variance', 'vi', 'var', 'v'],
    'se':       ['se', 'standard_error', 'stderr', 'se_es'],
    'n_total':  ['n_total', 'n', 'sample_size', 'total_n', 'sample_n']
}

# --- Geographic Column Synonyms ---
GEO_COLUMN_SPECS = {
    'latitude':  ['latitude', 'lat'],
    'longitude': ['longitude', 'lon', 'long', 'lng'],
    'country':   ['country', 'nation', 'region']
}

# Global placeholders
temp_raw_df = None
_data_type_widget = None
_mapping_container = None
# Ensure ANALYSIS_CONFIG always exists at module level so downstream cells
if 'ANALYSIS_CONFIG' not in globals():
    ANALYSIS_CONFIG = {}

# =============================================================================
# SAMPLE DATASETS
# ==================================================
BCG_DATA = {np.str_('trial'): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], 'id': ['Aronson', 'Ferguson & Simes', 'Rosenthal et al', 'Hart & Sutherland', 'Frimodt-Moller et al', 'Stein & Aronson', 'Vandiviere et al', 'TPT Madras', 'Coetzee & Berjak', 'Rosenthal et al', 'Comstock et al', 'Comstock & Webster', 'Comstock et al'], np.str_('year'): [1948, 1949, 1960, 1977, 1973, 1953, 1973, 1980, 1968, 1961, 1974, 1969, 1976], 'events_e': [4, 6, 3, 62, 33, 180, 8, 505, 29, 17, 186, 5, 27], 'nonevents_e': [119, 300, 228, 13536, 5036, 1361, 2537, 87886, 7470, 1699, 50448, 2493, 16886], 'events_c': [11, 29, 11, 248, 47, 372, 10, 499, 45, 65, 141, 3, 29], 'nonevents_c': [128, 274, 209, 12619, 5761, 1079, 619, 87892, 7232, 1600, 27197, 2338, 17825], np.str_('ablat'): [44, 55, 42, 52, 13, 44, 19, 13, 27, 42, 18, 33, 33], 'allocation': ['random', 'random', 'random', 'random', 'alternate', 'alternate', 'random', 'random', 'random', 'systematic', 'systematic', 'systematic', 'systematic']}
NORMAND_DATA = {np.str_('study'): [1, 2, 3, 4, 5, 6, 7, 8, 9], 'id': ['Edinburgh', 'Orpington-Mild', 'Orpington-Moderate', 'Orpington-Severe', 'Montreal-Home', 'Montreal-Transfer', 'Newcastle', 'Umea', 'Uppsala'], 'ne': [155, 31, 75, 18, 8, 57, 34, 110, 60], 'xe': [55, 27, 64, 66, 14, 19, 52, 21, 30], 'sde': [47, 7, 17, 20, 8, 7, 45, 16, 27], 'nc': [156, 32, 71, 18, 13, 52, 33, 183, 52], 'xc': [75, 29, 119, 137, 18, 18, 41, 31, 23], 'sdc': [64, 4, 29, 48, 11, 4, 34, 27, 20]}
KONST_DATA = {'id': [11, 11, 11, 11, 12, 12, 12, 12, 18, 18, 18, 27, 27, 27, 27, 56, 56, 56, 56, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 71, 71, 71, 86, 86, 86, 86, 86, 86, 86, 86, 91, 91, 91, 91, 91, 91, 108, 108, 108, 108, 108, 644, 644, 644, 644], 'school_id': [1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 1, 2, 3, 1, 2, 3, 4, 5, 6, 7, 8, 1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 1, 2, 3, 4], np.str_('study'): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56], np.str_('year'): [1976, 1976, 1976, 1976, 1989, 1989, 1989, 1989, 1994, 1994, 1994, 1976, 1976, 1976, 1976, 1997, 1997, 1997, 1997, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 1995, 1995, 1994, 1994], np.str_('yi'): [-0.18, -0.22, 0.23, -0.3, 0.13, -0.26, 0.19, 0.32, 0.45, 0.38, 0.29, 0.16, 0.65, 0.36, 0.6, 0.08, 0.04, 0.19, -0.06, -0.18, 0.0, 0.0, -0.28, -0.04, -0.3, 0.07, 0.0, 0.05, -0.08, -0.09, 0.3, 0.98, 1.19, -0.07, -0.05, -0.01, 0.02, -0.03, 0.0, 0.01, -0.1, 0.5, 0.66, 0.2, 0.0, 0.05, 0.07, -0.52, 0.7, -0.03, 0.27, -0.34, 0.12, 0.61, 0.04, -0.05], np.str_('vi'): [0.118, 0.118, 0.144, 0.144, 0.014, 0.014, 0.015, 0.024, 0.023, 0.043, 0.012, 0.02, 0.004, 0.004, 0.007, 0.019, 0.007, 0.005, 0.004, 0.02, 0.018, 0.019, 0.022, 0.02, 0.021, 0.006, 0.007, 0.007, 0.007, 0.007, 0.015, 0.011, 0.01, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.01, 0.011, 0.01, 0.009, 0.013, 0.013, 0.031, 0.031, 0.03, 0.03, 0.03, 0.087, 0.082, 0.067, 0.067]}
RAUDENBUSH_DATA = {'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], np.str_('author'): ['Rosenthal et al.', 'Conn et al.', 'Jose & Cody', 'Pellegrini & Hicks', 'Pellegrini & Hicks', 'Evans & Rosenthal', 'Fielder et al.', 'Claiborn', 'Kester', 'Maxwell', 'Carter', 'Flowers', 'Keshock', 'Henrikson', 'Fine', 'Grieger', 'Rosenthal & Jacobson', 'Fleming & Anttonen', 'Ginsburg'], np.str_('year'): [1974, 1968, 1971, 1972, 1972, 1969, 1971, 1969, 1969, 1970, 1970, 1966, 1970, 1970, 1972, 1970, 1968, 1971, 1970], np.str_('weeks'): [2, 21, 19, 0, 0, 3, 17, 24, 0, 1, 0, 0, 1, 2, 17, 5, 1, 2, 7], np.str_('setting'): ['group', 'group', 'group', 'group', 'group', 'group', 'group', 'group', 'group', 'indiv', 'group', 'group', 'indiv', 'indiv', 'group', 'group', 'group', 'group', 'group'], np.str_('tester'): ['aware', 'aware', 'aware', 'aware', 'blind', 'aware', 'blind', 'aware', 'aware', 'blind', 'blind', 'blind', 'blind', 'blind', 'aware', 'blind', 'aware', 'blind', 'aware'], np.str_('n1i'): [77, 60, 72, 11, 11, 129, 110, 26, 75, 32, 22, 43, 24, 19, 80, 72, 65, 233, 65], np.str_('n2i'): [339, 198, 72, 22, 22, 348, 636, 99, 74, 32, 22, 38, 24, 32, 79, 72, 255, 224, 67], np.str_('yi'): [0.03, 0.12, -0.14, 1.18, 0.26, -0.06, -0.02, -0.32, 0.27, 0.8, 0.54, 0.18, -0.02, 0.23, -0.18, -0.06, 0.3, 0.07, -0.07], np.str_('vi'): [0.0156, 0.0216, 0.0279, 0.1391, 0.1362, 0.0106, 0.0106, 0.0484, 0.0269, 0.063, 0.0912, 0.0497, 0.0835, 0.0841, 0.0253, 0.0279, 0.0193, 0.0088, 0.0303]}
CURTIS_DATA = {'row_id': [21, 22, 27, 32, 35, 38, 44, 63, 86, 87, 95, 96, 120, 125, 130, 163, 170, 182, 208, 209, 230, 231, 236, 242, 254, 256, 257, 267, 277, 287, 296, 306, 313, 320, 327, 334, 343, 344, 392, 393, 394, 395, 410, 411, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 479, 480, 523, 524, 525, 526, 547, 548, 561, 562, 575, 576, 614, 615, 623, 624, 636, 637, 721, 725, 726, 727, 731, 732, 733, 737, 738, 739, 743, 744, 745, 749, 750, 751, 755, 756, 757, 765, 774, 783], 'id': [44, 44, 121, 121, 121, 121, 159, 183, 209, 209, 210, 210, 290, 290, 290, 468, 470, 503, 505, 505, 506, 506, 510, 510, 550, 553, 553, 582, 582, 582, 582, 582, 596, 596, 596, 596, 666, 666, 2003, 2003, 2003, 2003, 2026, 2026, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2039, 2039, 2045, 2045, 2045, 2045, 2048, 2048, 2048, 2048, 2048, 2048, 2110, 2110, 2117, 2117, 2117, 2117, 2217, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2224, 2224, 2224], np.str_('genus'): ['ALNUS', 'ALNUS', 'ACER', 'QUERCUS', 'MALUS', 'ACER', 'CASTANEA', 'CITRUS', 'CASTANEA', 'CASTANEA', 'CASTANEA', 'CASTANEA', 'PINUS', 'NOTHOFAGUS', 'PSEUDOTSUGA', 'CASTANEA', 'CASTANEA', 'PINUS', 'QUERCUS', 'QUERCUS', 'LIRIODENDRON', 'LIRIODENDRON', 'QUERCUS', 'PINUS', 'BETULA', 'PICEA', 'PICEA', 'PIPER', 'SENNA', 'MYRIOCARPA', 'CECROPIA', 'TRICHOSPERMUM', 'BETULA', 'BETULA', 'BETULA', 'BETULA', 'PINUS', 'PINUS', 'BETULA', 'BETULA', 'BETULA', 'BETULA', 'PINUS', 'PINUS', 'ACER', 'ACER', 'ACER', 'QUERCUS', 'QUERCUS', 'QUERCUS', 'ACER', 'ACER', 'ACER', 'BETULA', 'BETULA', 'BETULA', 'FRAXINUS', 'FRAXINUS', 'FRAXINUS', 'BETULA', 'BETULA', 'BETULA', 'POPULUSX', 'POPULUSX', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PINUS', 'PINUS', 'POPULUS', 'POPULUS', 'PICEA', 'PICEA', 'BETULA', 'BETULA', 'MARANTHES', 'ACER', 'ACER', 'ACER', 'QUERCUS', 'QUERCUS', 'QUERCUS', 'ACER', 'ACER', 'ACER', 'BETULA', 'BETULA', 'BETULA', 'FRAXINUS', 'FRAXINUS', 'FRAXINUS', 'BETULA', 'BETULA', 'BETULA', 'QUERCUS', 'ACER', 'POPULUS'], np.str_('species'): ['RUBRA', 'RUBRA', 'RUBRUM', 'PRINUS', 'DOMESTICA', 'SACCHARINUM', 'SATIVA', 'SINENSIS', 'SATIVA', 'SATIVA', 'SATIVA', 'SATIVA', 'RADIATA', 'FUSCA', 'MENZIESII', 'SATIVA', 'SATIVA', 'ECHINATA', 'ALBA', 'ALBA', 'TULIPIFERA', 'TULIPIFERA', 'ALBA', 'ECHINATA', 'PENDULA', 'ABIES', 'ABIES', 'AURITUM', 'MULTIJUGA', 'LONGIPES', 'OBTUSIFOLIA', 'MEXICANUM', 'LENTA', 'PAPYRIFERA', 'POPULIFOLIA', 'ALLEGHANIENSIS', 'BANKSIANA', 'BANKSIANA', 'PUBESCENS', 'PUBESCENS', 'PUBESCENS', 'PUBESCENS', 'PONDEROSA', 'PONDEROSA', 'RUBRUM', 'RUBRUM', 'RUBRUM', 'RUBRA', 'RUBRA', 'RUBRA', 'PENSYLVANICUM', 'PENSYLVANICUM', 'PENSYLVANICUM', 'POPULIFOLIA', 'POPULIFOLIA', 'POPULIFOLIA', 'AMERICANA', 'AMERICANA', 'AMERICANA', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'EURAMERICANA', 'EURAMERICANA', 'MARIANA', 'MARIANA', 'MARIANA', 'MARIANA', 'GLAUCA', 'GLAUCA', 'MARIANA', 'MARIANA', 'BANKSIANA', 'BANKSIANA', 'EURAMERICANA', 'EURAMERICANA', 'ABIES', 'ABIES', 'PENDULA', 'PENDULA', 'CORYMBOSA', 'RUBRUM', 'RUBRUM', 'RUBRUM', 'RUBRA', 'RUBRA', 'RUBRA', 'PENSYLVANICUM', 'PENSYLVANICUM', 'PENSYLVANICUM', 'POPULIFOLIA', 'POPULIFOLIA', 'POPULIFOLIA', 'AMERICANA', 'AMERICANA', 'AMERICANA', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'RUBRA', 'SACCHARUM', 'TREMULOIDES'], np.str_('fungrp'): ['N2FIX', 'N2FIX', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'ANGIO', 'GYMNO', 'ANGIO', 'ANGIO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO'], np.str_('co2.ambi'): [350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 395.0, 350.0, 350.0, 350.0, 350.0, 340.0, 340.0, 340.0, 350.0, 350.0, 368.0, 389.0, 389.0, 371.0, 371.0, 360.0, 360.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 34.5, 34.5, 340.0, 340.0, 340.0, 340.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 34.5, 34.5, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 385.0, 385.0, 385.0], np.str_('co2.elev'): [650.0, 650.0, 700.0, 700.0, 700.0, 700.0, 700.0, 795.0, 700.0, 700.0, 700.0, 700.0, 640.0, 640.0, 640.0, 700.0, 700.0, 695.0, 793.0, 793.0, 787.0, 787.0, 700.0, 700.0, 700.0, 750.0, 750.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 750.0, 750.0, 700.0, 700.0, 560.0, 560.0, 650.0, 650.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 69.3, 69.3, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 69.3, 69.3, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 642.0, 642.0, 642.0], np.str_('units'): ['ul/l', 'ul/l', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ul/l', 'ul/l', 'ul/l', 'umol/mol', 'ppm', 'ul/l', 'cm3/m3', 'cm3/m3', 'ppm', 'ppm', 'ul/l', 'ul/l', 'umol/mol', 'cm3/m3', 'cm3/m3', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'ubar', 'ubar', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'Pa', 'Pa', 'ppm', 'ppm', 'ppm', 'ppm', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'Pa', 'Pa', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l'], np.str_('time'): [47, 47, 59, 70, 64, 50, 730, 365, 365, 365, 120, 120, 120, 120, 120, 1095, 180, 287, 168, 168, 168, 168, 210, 168, 40, 180, 180, 111, 111, 111, 111, 111, 90, 90, 90, 90, 270, 270, 34, 34, 35, 35, 60, 60, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 148, 148, 155, 155, 155, 155, 112, 112, 112, 112, 112, 112, 158, 158, 47, 47, 35, 35, 210, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 60, 60, 60], np.str_('pot'): ['0.5', '0.5', '2.6', '2.6', '2.6', '2.6', 'GRND', '9', '24', '24', '12', '12', '4', '4', '4', '24', '24', '0.95', '2.6', '2.6', '3.5', '3.5', '0.95', '0.95', 'HYDRO', '50', '50', '0.67', '0.67', '0.67', '0.67', '0.67', '0.66', '0.66', '0.66', '0.66', '0.04', '0.04', '0.2', '0.2', '0.3', '0.3', '1.8', '1.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', 'GRND', 'GRND', '0.164', '0.164', '0.164', '0.164', '0.2', '0.2', '0.2', '0.2', '0.2', '0.2', 'GRND', 'GRND', '0.2', '0.2', '0.2', '0.2', '10', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '6', '6', '6'], np.str_('method'): ['GC', 'GC', 'GH', 'GH', 'GH', 'GH', 'GC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GC', 'GC', 'GC', 'OTC', 'OTC', 'GC', 'GH', 'GH', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'OTC', 'OTC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'OTC', 'OTC', 'GH', 'GH', 'GH', 'GH', 'OTC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GC', 'GC', 'GC'], np.str_('stock'): ['SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SAP', 'SAP', 'SAP', 'SAP', 'SAP', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED'], np.str_('xtrt'): ['FERT', 'FERT', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'FERT', 'FERT', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'FERT', 'FERT', 'FERT', 'FERT', 'NONE', 'NONE', 'NONE', 'OZONE', 'OZONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'UVB', 'UVB', 'TEMP', 'TEMP', 'OZONE', 'OZONE', 'TEMP', 'TEMP', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'FERT', 'H2O', 'H2O', 'FERT', 'FERT', 'UVB', 'UVB', 'UVB', 'UVB', 'UVB', 'UVB', 'FERT', 'FERT', 'TEMP', 'TEMP', 'TEMP', 'TEMP', 'NONE', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'NONE', 'NONE', 'NONE'], np.str_('level'): ['HIGH', 'CONTROL', '.', '.', '.', '.', '.', '.', 'HIGH', 'CONTROL', '.', '.', '.', '.', '.', '.', '.', '.', 'HIGH', 'CONTROL', 'HIGH', 'CONTROL', '.', '.', '.', 'HIGH', 'LOW', '.', '.', '.', '.', '.', '.', '.', '.', '.', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'HIGH', 'LOW', 'WW', 'DRT', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', '.', '.', '.'], 'xe': [6.8168999999999995, 2.5961, 2.99, 5.91, 4.61, 10.78, 153.5, 1439.0, 183.6, 71.72, 45.8, 72.5, 59.1, 4.9, 19.0, 146.0, 129.0, 6.91, 17.89, 14.68, 63.97, 6.01, 14.15, 2.2, 23.11, 262.1, 273.3, 10.14, 15.72, 9.01, 21.33, 14.68, 8.36, 7.91, 12.45, 7.84, 0.09315999999999999, 0.59046, 4.77, 3.74, 3.15, 3.78, 2.15, 1.76, 47.38, 258.53, 482.04, 30.59, 172.4, 277.46, 23.08, 196.8, 254.39, 31.1, 186.0, 232.37, 25.19, 120.8, 262.77, 38.36, 178.33, 186.26, 113.0, 75.0, 4.773, 3.272, 5.249, 2.798, 0.333, 0.64, 0.498, 0.824, 0.843, 1.533, 562.8, 374.4, 1.092, 1.12, 6.91, 6.94, 16.959, 2.4, 16.98, 28.18, 3.09, 13.8, 18.2, 1.9, 10.23, 16.59, 2.34, 16.22, 24.55, 1.51, 9.12, 18.62, 1.9, 14.45, 16.98, 16.9, 7.2, 102.6], 'sde': [1.769982, 0.6674662, 0.856, 1.742, 1.407, 1.163, 27.1932, 142.028, 39.06, 14.3, 15.8, 15.0, 12.9, 1.9, 5.4, 10.0, 59.0, 3.22, 10.185, 5.013, 7.66, 1.02, 8.6467, 0.4111, 0.8443, 16.4, 14.0, 2.52, 2.76, 2.58, 3.06, 2.82, 3.915, 1.809, 6.944, 3.324, 0.071404, 0.41036900000000004, 0.198, 0.0424, 0.297, 0.424, 0.36, 0.28, 5.015, 50.315, 14.801, 2.152, 23.678, 14.834, 4.294, 26.614, 32.624, 1.073, 17.729, 8.867, 2.151, 20.773, 17.832, 3.215, 11.833, 11.834, 4.472, 8.944, 2.1477, 0.9215, 1.6628, 0.5958, 0.147, 0.154, 0.163, 0.234, 0.234, 0.395, 62.3863, 109.5673, 0.0297, 0.0141, 0.3111, 0.4243, 2.186, 0.5634, 1.9596, 3.2333, 0.7348, 2.425, 3.1843, 0.1225, 0.6124, 2.9149, 0.2694, 2.8414, 1.3962, 0.1715, 1.5922, 2.1556, 0.2205, 1.6657, 0.5389, 1.7321, 1.7321, 6.2354], 'ne': [3, 5, 5, 5, 4, 5, 3, 3, 20, 16, 20, 24, 8, 8, 8, 5, 12, 5, 5, 5, 5, 5, 6, 10, 22, 6, 6, 4, 4, 4, 4, 4, 10, 10, 10, 10, 10, 10, 2, 2, 2, 2, 16, 16, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 48, 48, 48, 48, 10, 10, 10, 10, 10, 10, 5, 5, 2, 2, 2, 2, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3], 'xc': [3.945, 2.2512, 1.93, 6.62, 4.1, 6.42, 127.3, 1140.6, 144.1, 59.94, 28.2, 60.5, 46.4, 4.2, 18.5, 136.0, 90.4, 6.81, 14.62, 11.18, 54.03, 4.93, 8.2, 1.33, 14.94, 235.5, 236.4, 8.81, 13.22, 9.8, 18.6, 12.18, 5.3, 3.6, 4.98, 4.01, 0.07382, 0.41117000000000004, 3.83, 3.4, 2.4, 3.17, 1.7, 1.66, 32.99, 221.0, 396.31, 44.8, 44.93, 187.67, 18.8, 109.07, 168.88, 29.36, 161.05, 232.69, 23.2, 114.68, 168.74, 26.49, 161.74, 180.48, 76.0, 60.0, 3.862, 2.49, 4.031, 2.35, 0.243, 0.443, 0.328, 0.55, 0.599, 0.842, 381.6, 298.2, 0.971, 0.975, 5.95, 5.78, 10.578, 2.24, 10.96, 19.5, 2.04, 4.36, 14.79, 1.51, 5.89, 13.8, 2.24, 11.48, 19.95, 2.14, 5.49, 13.49, 1.48, 8.71, 16.59, 7.2, 4.6, 69.7], 'sdc': [1.115797, 0.32758390000000004, 0.552, 1.631, 1.257, 2.026, 47.4582, 82.965, 25.7, 14.28, 11.9, 14.0, 13.1, 1.0, 8.1, 35.0, 36.0, 1.57, 2.294, 5.749, 1.62, 1.35, 3.3558, 0.1897, 0.8443, 15.0, 7.5, 2.06, 1.94, 2.06, 2.34, 2.46, 6.625, 3.026, 3.937, 2.106, 0.08329399999999999, 0.255702, 0.1414, 0.0566, 0.014, 0.424, 0.32, 0.28, 5.375, 32.542, 41.697, 7.879, 11.845, 26.635, 4.668, 29.574, 20.734, 3.943, 8.876, 17.897, 1.078, 26.625, 26.83, 1.79, 14.856, 11.867, 6.708, 8.944, 1.4272, 0.6582, 1.2956, 0.582, 0.118, 0.13, 0.127, 0.112, 0.132, 0.151, 64.3988, 96.3745, 0.0255, 0.0127, 0.198, 0.099, 1.21, 0.3919, 1.9351, 2.2535, 0.3674, 2.474, 3.5028, 0.1715, 1.3962, 1.5922, 0.3919, 2.0086, 2.3025, 0.4899, 1.3227, 0.7593, 0.1715, 1.5187, 0.9553, 2.5981, 1.7321, 3.6373], 'nc': [5, 5, 5, 5, 4, 3, 3, 3, 20, 16, 23, 24, 8, 8, 8, 5, 12, 5, 5, 5, 5, 5, 6, 10, 22, 6, 6, 4, 4, 4, 4, 4, 10, 10, 10, 10, 10, 10, 2, 2, 2, 2, 16, 16, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 48, 48, 48, 48, 10, 10, 10, 10, 10, 10, 5, 5, 2, 2, 2, 2, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3]}

BUILT_IN_DATASETS = {
    # Existing Datasets
    'Binary (BCG Vaccine - Tuberculosis)': {'data': BCG_DATA, 'type': 'raw_binary'},
    'Continuous (Normand 1999 - Stroke Rehab)': {'data': NORMAND_DATA, 'type': 'raw_continuous'},
    '3-Level Pre-Calculated (Konstantopoulos 2011)': {'data': KONST_DATA, 'type': 'pre_calculated'},
    'Meta-Regression / Splines (Raudenbush 1985)': {'data': RAUDENBUSH_DATA, 'type': 'pre_calculated'},
    'Ecology Continuous (Curtis 1998 - Plant CO2)': {'data': CURTIS_DATA, 'type': 'raw_continuous'}
}
# =============================================================================
# HELPER: UNIVERSAL FILE EXTRACTOR
# =============================================================================
def get_uploaded_file_data(uploader_widget):
    """Robustly extracts filename and binary content from ipywidgets.FileUpload."""
    val = uploader_widget.value
    if not val:
        return None, None
    try:
        if isinstance(val, (tuple, list)):
            file_obj = val[0]
            fname = file_obj['name']
            content = file_obj['content']
        elif isinstance(val, dict):
            keys = list(val.keys())
            if not keys:
                return None, None
            fname = keys[0]
            content = val[fname]['content']
        else:
            raise ValueError(f"Unknown widget format: {type(val)}")

        if hasattr(content, 'tobytes'):
            content = content.tobytes()
        return fname, content

    except Exception as e:
        print(f"Debug Info - Raw uploader value type: {type(val)}")
        raise e


# =============================================================================
# HELPER: DUPLICATE COLUMN
# =============================================================================
def _check_duplicate_columns(df, source_label="file"):
    """
    Raises ValueError if the DataFrame has duplicate column names, listing
    the offenders so the user knows exactly what to fix.
    """
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
        raise ValueError(
            f"The {source_label} contains duplicate column names: "
            f"{dupes}. "
            f"Please rename them in the source file so each column is unique, "
            f"then re-upload."
        )


# =============================================================================
# HELPER: NUMERIC COLUMN VALIDATOR FOR FINALIZE
# =============================================================================

# Columns that MUST be coercible to numeric for each data type.
# Any NaN produced by coercion on these columns = a dropped row.
_NUMERIC_REQUIRED = {
    'raw_continuous': ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc'],
    'raw_binary':     ['events_e', 'nonevents_e', 'events_c', 'nonevents_c'],
    'pre_calculated': ['yi'],           # variance/se handled separately below
}
_NUMERIC_SOFT = {
    # soft = coerce + warn but do NOT drop rows, downstream can decide
    'pre_calculated': ['variance', 'se', 'n_total'],
}

def _validate_and_coerce_mapped_numerics(df, col_map, data_type):
    """
    For each mapped column that is required to be numeric:
      1. Coerces the column to numeric (errors → NaN).
      2. Counts how many rows became NaN.
      3. Collects per-column warnings.
      4. Raises ValueError if any HARD-required column has > 0 NaN rows
         (after giving the user a clear breakdown).

    For soft-required columns (variance, se, n_total in pre_calculated mode)
    it warns but does not raise.

    Parameters
    ----------
    df       : DataFrame already renamed to standard column names
    col_map  : {std_name: original_col_name} — used only for the error message
               so the user sees their own column name, not the internal one
    data_type: 'raw_continuous' | 'raw_binary' | 'pre_calculated'

    Returns
    -------
    df       : DataFrame with numeric columns coerced
    warn_html: HTML string (empty string if no warnings)
    """
    df = df.copy()
    hard_fields = _NUMERIC_REQUIRED.get(data_type, [])
    soft_fields = _NUMERIC_SOFT.get(data_type, [])

    hard_errors = []   # will block finalize
    soft_warnings = [] # will display but allow through

    for std_name in hard_fields:
        if std_name not in df.columns:
            continue
        original_name = col_map.get(std_name, std_name)
        before = df[std_name].notna().sum()
        df[std_name] = pd.to_numeric(df[std_name], errors='coerce')
        after  = df[std_name].notna().sum()
        lost   = int(before - after)
        if lost > 0:
            # Identify which rows so the user can fix the source
            bad_idx = df.index[df[std_name].isna()].tolist()
            # Try to show study IDs if available, otherwise row numbers
            if 'id' in df.columns:
                bad_labels = df.loc[bad_idx, 'id'].astype(str).tolist()
                label_str  = ", ".join(bad_labels[:6])
                if len(bad_labels) > 6:
                    label_str += f" … +{len(bad_labels)-6} more"
            else:
                label_str = ", ".join(f"row {i+2}" for i in bad_idx[:6])  # +2: 1-indexed + header
                if len(bad_idx) > 6:
                    label_str += f" … +{len(bad_idx)-6} more"

            hard_errors.append(
                f"<li style='margin-bottom:6px;'>"
                f"<b>{original_name}</b> → <code>{std_name}</code>: "
                f"<b>{lost}</b> non-numeric value(s) found. "
                f"Affected studies/rows: <i>{label_str}</i>."
                f"</li>"
            )

    for std_name in soft_fields:
        if std_name not in df.columns:
            continue
        original_name = col_map.get(std_name, std_name)
        before = df[std_name].notna().sum()
        df[std_name] = pd.to_numeric(df[std_name], errors='coerce')
        after  = df[std_name].notna().sum()
        lost   = int(before - after)
        if lost > 0:
            soft_warnings.append(
                f"<li style='margin-bottom:6px;'>"
                f"<b>{original_name}</b> → <code>{std_name}</code>: "
                f"<b>{lost}</b> non-numeric value(s) coerced to NaN. "
                f"These rows will have missing variance/SE — check downstream diagnostics."
                f"</li>"
            )

    # --- Build HTML output ---
    warn_html = ""

    if hard_errors:
        items = "".join(hard_errors)
        # Raise with HTML so the caller can display it properly
        err_html = (
            f"<div style='background-color:#f8d7da; color:#721c24; padding:12px 15px; "
            f"border-radius:6px; border:1px solid #f5c6cb; margin-top:10px;'>"
            f"<b>❌ Non-numeric values in required columns</b><br>"
            f"<small>The following mapped columns contain text or missing values where numbers "
            f"are required. Please fix the source data and re-upload, or remove those rows.</small>"
            f"<ul style='margin:8px 0 0 0; padding-left:18px;'>{items}</ul>"
            f"</div>"
        )
        raise ValueError(err_html)   # caller displays this as HTML, not plain text

    if soft_warnings:
        items = "".join(soft_warnings)
        warn_html = (
            f"<div style='background-color:#fff3cd; color:#856404; padding:12px 15px; "
            f"border-radius:6px; border:1px solid #ffc107; margin-top:10px;'>"
            f"<b>⚠️ Non-numeric values in optional columns</b>"
            f"<ul style='margin:8px 0 0 0; padding-left:18px;'>{items}</ul>"
            f"</div>"
        )

    return df, warn_html



# =============================================================================
# HELPER: COERCE NUMERIC COLUMNS
# =============================================================================
def _coerce_numeric_columns(df):
    """
    Google Sheets returns everything as strings. Converts columns that are
    predominantly numeric to numeric types.

    Defensive changes vs original:
      - Skips columns whose name contains 'id', 'name', 'label', 'author',
        'study', 'paper' (case-insensitive) to avoid corrupting ID fields.
      - Raises threshold to 0.8 to reduce false-positive coercions.
    """
    _ID_LIKE = {'id', 'name', 'label', 'author', 'study', 'paper', 'title'}
    df = df.copy()
    for col in df.columns:
        # Skip columns that are already numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        # Skip obvious identifier columns
        col_lower = str(col).lower().strip()
        if any(tok in col_lower for tok in _ID_LIKE):
            continue
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            non_empty = df[col].replace('', np.nan).dropna()
            if len(non_empty) > 0:
                valid_ratio = converted.notna().sum() / len(non_empty)
                if valid_ratio >= 0.8:
                    df[col] = converted
        except Exception:
            pass
    return df

# -----------------------------------------------------------------------------
# HELPER: Error display
# -----------------------------------------------------------------------------

def _show_finalize_error(e):
    """Displays a user-friendly error and prints the full traceback to stdout."""
    import traceback
    tb = traceback.format_exc()
    display(HTML(
        f"<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
        f"color:#721c24; border-radius:4px; margin-top:10px;'>"
        f"❌ <b>Mapping Error:</b> {e}"
        f"<br><small style='color:#999;'>See cell output below for full traceback.</small></div>"
    ))
    print(tb)

# -----------------------------------------------------------------------------
# HELPER: Safe CSV reader with encoding fallback# and duplicate-column detection.
# -----------------------------------------------------------------------------
def _safe_read_csv(content_bytes):
    """
    Attempts to parse CSV bytes with UTF-8, then falls back to common
    Western encodings.  Also detects duplicate column names and warns.
    Returns a DataFrame.
    """
    _ENCODINGS = ['utf-8', 'latin-1', 'windows-1252', 'utf-8-sig']
    last_exc = None
    for enc in _ENCODINGS:
        try:
            df = pd.read_csv(io.BytesIO(content_bytes), encoding=enc)
            # P8 — duplicate column check
            if df.columns.duplicated().any():
                dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
                display(HTML(
                    f"<div style='padding:10px; background-color:#fff3cd; border-left:4px solid #ffc107; "
                    f"color:#856404; margin-bottom:10px;'>"
                    f"⚠️ <b>Duplicate column names detected:</b> {dupes}. "
                    f"Pandas has auto-renamed them (e.g. <code>col</code> → <code>col.1</code>). "
                    f"Consider fixing the source file for reliable mapping.</div>"
                ))
            return df
        except UnicodeDecodeError as e:
            last_exc = e
            continue
    raise ValueError(
        f"Could not decode the CSV file with any of the attempted encodings "
        f"({', '.join(_ENCODINGS)}). Last error: {last_exc}"
    )

# =============================================================================
# HELPER: VALIDATE GEOGRAPHIC DATA & BUILD WARNINGS
# =============================================================================
def _validate_geo_data(df, geo_map):
    """
    Validates mapped geographic columns and returns:
      - warnings_html: HTML string with per-row warnings
      - summary_html:  HTML string with overall geo summary
      - geo_type:      'coordinates' | 'country' | 'both' | None
    """
    warnings = []
    has_coords = ('latitude' in geo_map and 'latitude' in df.columns
                  and 'longitude' in geo_map and 'longitude' in df.columns)
    has_country = ('country' in geo_map and 'country' in df.columns)

    if not has_coords and not has_country:
        return "", "", None

    geo_type_parts = []
    if has_coords:
        geo_type_parts.append('coordinates')
    if has_country:
        geo_type_parts.append('country')
    geo_type = '+'.join(geo_type_parts) if len(geo_type_parts) == 2 else geo_type_parts[0]

    coord_missing = 0
    country_missing = 0

    if has_coords:
        lat_s = pd.to_numeric(df['latitude'], errors='coerce')
        lon_s = pd.to_numeric(df['longitude'], errors='coerce')
        coord_missing = (lat_s.isna() | lon_s.isna()).sum()

        if coord_missing > 0:
            missing_ids = df.loc[lat_s.isna() | lon_s.isna(), 'id'].tolist() if 'id' in df.columns else []
            id_hint = f" (studies: {', '.join(str(s) for s in missing_ids[:5])}{'…' if len(missing_ids)>5 else ''})" if missing_ids else ""
            warnings.append(f"⚠️ <b>{coord_missing}</b> row(s) have missing or non-numeric coordinates{id_hint}. These will be excluded from the geographic map.")

        lat_oob = ((lat_s < -90) | (lat_s > 90)).sum()
        lon_oob = ((lon_s < -180) | (lon_s > 180)).sum()
        if lat_oob > 0:
            warnings.append(f"⚠️ <b>{lat_oob}</b> latitude value(s) are outside the valid range (−90 to 90).")
        if lon_oob > 0:
            warnings.append(f"⚠️ <b>{lon_oob}</b> longitude value(s) are outside the valid range (−180 to 180).")

    if has_country:
        country_s = df['country'].astype(str).str.strip().replace('', np.nan)
        country_missing = country_s.isna().sum() + (country_s == 'nan').sum()
        if country_missing > 0:
            warnings.append(f"⚠️ <b>{country_missing}</b> row(s) have missing country/region values. These will be excluded from country-level map layers.")

    warnings_html = ""
    if warnings:
        items = "".join(f"<li style='margin-bottom:4px;'>{w}</li>" for w in warnings)
        warnings_html = f"""
        <div style='background-color:#fff3cd; color:#856404; padding:10px 15px; border-radius:6px;
                    border:1px solid #ffc107; margin-top:10px; margin-bottom:10px;'>
            <b>🌍 Geographic Data Warnings</b>
            <ul style='margin:6px 0 0 0; padding-left:18px;'>{items}</ul>
        </div>
        """

    parts = []
    if has_coords:
        valid_coords = len(df) - (coord_missing if has_coords else 0)
        parts.append(f"Coordinates: <b>{valid_coords}</b>/{len(df)} valid")
    if has_country:
        valid_countries = len(df) - country_missing
        n_unique = country_s.nunique()
        parts.append(f"Countries: <b>{valid_countries}</b>/{len(df)} valid ({n_unique} unique)")
    summary_html = "<br>🌍 Geographic data: " + " &nbsp;·&nbsp; ".join(parts) if parts else ""

    return warnings_html, summary_html, geo_type


# =============================================================================
# HELPER: RICH MODERATOR SUMMARY
# =============================================================================
def _build_moderator_summary_html(df, mapped_cols, extra_cols, mode_label,
                                   extra_info="", geo_warnings_html="",
                                   geo_summary_html="", geo_type=None):
    """Builds a rich HTML summary showing moderators, their unique values, and record counts."""
    n_rows = len(df)
    n_mapped = len(mapped_cols)
    geo_badge = f" &nbsp;·&nbsp; 🌍 Geo: <b>{geo_type}</b>" if geo_type else ""

    html = f"""
    <div style='background-color:#d4edda; color:#155724; padding:15px; border-radius:8px;
                border:1px solid #c3e6cb; margin-bottom:15px;'>
        <h4 style='margin:0 0 8px 0;'>✅ Data Ready ({mode_label})</h4>
        <b>{n_rows}</b> rows loaded &nbsp;·&nbsp;
        <b>{n_mapped}</b> mapped columns &nbsp;·&nbsp;
        <b>{len(extra_cols)}</b> moderator columns detected{geo_badge}
        {extra_info}
        {geo_summary_html}
    </div>
    """

    html += geo_warnings_html

    if extra_cols:
        html += """
        <div style='background-color:#f8f9fa; padding:12px; border-radius:8px;
                    border:1px solid #dee2e6; margin-bottom:15px;'>
            <h4 style='color:#2E86AB; margin:0 0 10px 0;'>📊 Moderator Variables Overview</h4>
            <table style='width:100%; border-collapse:collapse; font-size:13px;'>
                <thead>
                    <tr style='border-bottom:2px solid #dee2e6; text-align:left;'>
                        <th style='padding:6px 10px;'>Column</th>
                        <th style='padding:6px 10px;'>Type</th>
                        <th style='padding:6px 10px;'>Unique</th>
                        <th style='padding:6px 10px;'>Missing</th>
                        <th style='padding:6px 10px;'>Values (count)</th>
                    </tr>
                </thead>
                <tbody>
        """
        MAX_VALUES_SHOWN = 8

        for col in extra_cols:
            series = df[col]
            n_unique = series.nunique()
            n_missing = series.isna().sum() + (series.astype(str).str.strip() == '').sum()
            is_numeric = pd.api.types.is_numeric_dtype(series)
            col_type = "Numeric" if is_numeric else "Categorical"

            if is_numeric:
                clean = pd.to_numeric(series, errors='coerce').dropna()
                if len(clean) > 0:
                    val_summary = f"Range: {clean.min():.3g} – {clean.max():.3g}, Mean: {clean.mean():.3g}"
                else:
                    val_summary = "<i>No valid numeric data</i>"
            else:
                counts = series.value_counts(dropna=True)
                parts = []
                for val, cnt in counts.head(MAX_VALUES_SHOWN).items():
                    val_str = str(val).strip()
                    if len(val_str) > 30:
                        val_str = val_str[:27] + "..."
                    parts.append(f"<b>{val_str}</b>&nbsp;({cnt})")
                val_summary = ", &nbsp;".join(parts)
                if len(counts) > MAX_VALUES_SHOWN:
                    val_summary += f", &nbsp;<i>… +{len(counts) - MAX_VALUES_SHOWN} more</i>"

            miss_style = "color:#c0392b; font-weight:bold;" if n_missing > 0 else ""

            html += f"""
                <tr style='border-bottom:1px solid #eee;'>
                    <td style='padding:6px 10px; font-weight:600;'>{col}</td>
                    <td style='padding:6px 10px;'>{col_type}</td>
                    <td style='padding:6px 10px;'>{n_unique}</td>
                    <td style='padding:6px 10px; {miss_style}'>{n_missing}</td>
                    <td style='padding:6px 10px; font-size:12px;'>{val_summary}</td>
                </tr>
            """

        html += """
                </tbody>
            </table>
        </div>
        """
    else:
        html += """
        <div style='background-color:#fff3cd; color:#856404; padding:10px; border-radius:5px;
                    border:1px solid #ffc107; margin-bottom:15px;'>
            ⚠️ No moderator columns detected beyond the mapped variables.
            If you expected moderator variables, verify that your spreadsheet includes them.
        </div>
        """

    html += "<p style='color:#555; font-size:12px;'>Please proceed to the next cell to configure analysis filters.</p>"
    return html


# =============================================================================
# HELPER: BUILD GEO MAPPING UI SECTION
# =============================================================================
def _build_geo_mapping_widgets(df):
    cols_lower = [str(c).lower().strip() for c in df.columns]
    options_with_none = ['None'] + list(df.columns)
    geo_widgets = {}

    GEO_FIELD_INFO = {
        'latitude':  {'label': 'Latitude:',  'desc': 'Decimal latitude (−90 to 90). Optional.'},
        'longitude': {'label': 'Longitude:', 'desc': 'Decimal longitude (−180 to 180). Optional.'},
        'country':   {'label': 'Country / Region:', 'desc': 'Country or region name. Optional.'}
    }

    ui_rows = [widgets.HTML("""
    <hr>
    <h4 style='color:#2E86AB; margin-top:15px;'>🌍 Geographic Columns <span style='font-size:12px; color:#888; font-weight:normal;'>(Optional)</span></h4>
    <div style='background-color:#e8f5e9; padding:10px; border-radius:5px; margin-bottom:10px; color:#2e7d32; font-size:13px;'>
        <b>Optional:</b> Map geographic columns to enable a publication-quality study location map in a later cell.<br>
        You can map <b>coordinates</b> (Lat + Lon), <b>country/region</b>, or <b>both</b>. Leave as "None" to skip.
    </div>
    """)]

    for std_name, synonyms in GEO_COLUMN_SPECS.items():
        info = GEO_FIELD_INFO[std_name]
        guessed_val = 'None'
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        w = widgets.Dropdown(
            options=options_with_none,
            value=guessed_val,
            description=info['label'],
            style={'description_width': '180px'},
            layout=widgets.Layout(width='600px')
        )
        geo_widgets[std_name] = w
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:185px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{info['desc']}</i></div>")
        ]))

    return widgets.VBox(ui_rows), geo_widgets

def _finalize_geo_columns(df, geo_widgets, col_map_values):
    geo_map = {}
    for std_name, w in geo_widgets.items():
        val = w.value
        if val != 'None':
            geo_map[std_name] = val

    has_lat = 'latitude' in geo_map
    has_lon = 'longitude' in geo_map
    if has_lat != has_lon:
        mapped_one = 'Latitude' if has_lat else 'Longitude'
        missing_one = 'Longitude' if has_lat else 'Latitude'
        raise ValueError(f"You mapped {mapped_one} but not {missing_one}. Please map both coordinates or neither.")

    if not geo_map:
        return geo_map, "", "", None

    for std_name, orig_col in geo_map.items():
        if orig_col in col_map_values:
            raise ValueError(f"Geographic column '{orig_col}' is already mapped to a core analysis field. Please choose a different column or set it to 'None'.")

    rename_map = {}
    for std_name, orig_col in geo_map.items():
        if orig_col != std_name:
            rename_map[orig_col] = std_name
    if rename_map:
        df.rename(columns=rename_map, inplace=True)

    resolved_geo_map = {std_name: std_name for std_name in geo_map}
    geo_warnings_html, geo_summary_html, geo_type = _validate_geo_data(df, resolved_geo_map)

    return geo_map, geo_warnings_html, geo_summary_html, geo_type


# =============================================================================
# UI PART 1: DATA LOADING (Tabs)
# =============================================================================
# --- Tab 1: Google Sheets ---
btn_auth = widgets.Button(description="1. Connect Google Account", button_style='warning', icon='google')
txt_sheet_name = widgets.Text(value='tesis', description='Sheet Name:', layout=widgets.Layout(width='300px'))
btn_fetch_ws = widgets.Button(description="2. Find Worksheets", button_style='primary', disabled=True)
dd_worksheets = widgets.Dropdown(description='Worksheet:', layout=widgets.Layout(width='300px'), disabled=True)
btn_load_sheet = widgets.Button(description="3. Load Data", button_style='success', disabled=True)

if globals().get('IS_COLAB', False):
    gs_vbox = widgets.VBox([
        widgets.HTML("<b>Step A: Load from Google Sheets</b>"),
        btn_auth,
        widgets.HBox([txt_sheet_name, btn_fetch_ws]),
        widgets.HBox([dd_worksheets, btn_load_sheet])
    ])
else:
    gs_vbox = widgets.VBox([
        widgets.HTML("""
        <div style='padding: 20px; background-color: #fff3cd; border-left: 5px solid #ffc107; border-radius: 4px; font-family: sans-serif;'>
            <h3 style='margin-top: 0; color: #856404;'>⚠️ Cloud Feature Only</h3>
            <p style='color: #856404; font-size: 14px;'>Google Sheets integration is only supported when running this tool online in Google Colaboratory.</p>
            <p style='color: #856404; font-size: 14px; margin-bottom: 0;'><b>Please click the "Upload Excel/CSV" tab above to load your local data.</b></p>
        </div>
        """)
    ])

# --- Tab 2: Local File ---
uploader = widgets.FileUpload(accept='.csv,.xlsx,.xls', multiple=False, description='Select Data')
dd_local_sheets = widgets.Dropdown(
    description='Select Sheet:',
    layout=widgets.Layout(width='300px', display='none'),
    style={'description_width': 'initial'}
)
btn_process_file = widgets.Button(description="Load Data File", button_style='success', disabled=True)
local_vbox = widgets.VBox([
    widgets.HTML(
        "<b>Step B: Upload Local File</b><br>"
        "1. Choose a .csv or .xlsx file.<br>"
        "2. Click 'Load Data File' to proceed."
    ),
    uploader,
    dd_local_sheets,
    btn_process_file
])

# --- Output Areas ---
log_output = widgets.Output()
step2_header_output = widgets.Output()
step2_body_output = widgets.Output()

# =============================================================================
# LOGIC PART 1: LOADERS
# =============================================================================
def on_auth_clicked(b):
    with log_output:
        clear_output(wait=True)
        display(HTML("<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-bottom: 10px;'>⏳ <b>Authenticating with Google...</b> Please follow the popup instructions.</div>"))
        try:
            auth.authenticate_user()
            creds, _ = default()
            global gc
            gc = gspread.authorize(creds)
            btn_auth.button_style = 'success'
            btn_auth.description = "Connected ✓"
            btn_auth.disabled = True
            btn_fetch_ws.disabled = False
            clear_output(wait=True)
            display(HTML("<div style='padding: 10px; background-color: #d4edda; border-left: 4px solid #28a745; color: #155724; margin-bottom: 10px;'>✅ <b>Authentication successful.</b> You may now find your worksheets.</div>"))
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-bottom: 10px;'>❌ <b>Authentication failed:</b> {e}</div>"))

def on_fetch_ws_clicked(b):
    with log_output:
        clear_output(wait=True)
        if 'gc' not in globals():
            display(HTML("<div style='padding: 10px; background-color: #fff3cd; border-left: 4px solid #ffc107; color: #856404; margin-bottom: 10px;'>⚠️ <b>Please connect your Google Account first.</b></div>"))
            return
        name = txt_sheet_name.value.strip()
        if not name:
            display(HTML("<div style='padding: 10px; background-color: #fff3cd; border-left: 4px solid #ffc107; color: #856404; margin-bottom: 10px;'>⚠️ <b>Please enter a valid Google Sheet name.</b></div>"))
            return
        display(HTML(f"<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-bottom: 10px;'>🔎 <b>Searching for spreadsheet:</b> '{name}'...</div>"))
        try:
            global spreadsheet
            spreadsheet = gc.open(name)
            titles = [ws.title for ws in spreadsheet.worksheets()]
            dd_worksheets.options = titles
            dd_worksheets.disabled = False
            btn_load_sheet.disabled = False
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>Found {len(titles)} worksheet(s).</b> Please select one and click 'Load Data'.</div>"))
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-bottom: 10px;'>❌ <b>Error locating spreadsheet:</b> Ensure the name is exact and the account has access. ({e})</div>"))

def on_load_sheet_clicked(b):
    df = None
    with log_output:
        clear_output(wait=True)
        display(HTML(
            f"<div style='padding:10px; background-color:#e2e3e5; border-left:4px solid #6c757d; "
            f"color:#383d41; margin-bottom:10px;'>📥 <b>Downloading worksheet:</b> "
            f"'{dd_worksheets.value}'...</div>"
        ))
        try:
            ws = spreadsheet.worksheet(dd_worksheets.value)
            rows = ws.get_all_values()
            if len(rows) < 2:
                raise ValueError("Worksheet is empty or lacks data rows below the header.")

            df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
            df.columns = [str(c).strip() for c in df.columns]

            # P6 — empty DataFrame guard
            if df.empty:
                raise ValueError(
                    "The worksheet has a header row but no data rows. "
                    "Please check the sheet content."
                )

            df = df.replace('', np.nan).dropna(how='all').dropna(axis=1, how='all').fillna('')
            df = _coerce_numeric_columns(df)    # uses patched version (P10)

            global temp_raw_df
            temp_raw_df = df
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:10px; background-color:#d4edda; border-left:4px solid #28a745; "
                f"color:#155724; margin-bottom:10px;'>✅ <b>Data successfully loaded!</b> "
                f"Analyzed {len(df)} rows and {len(df.columns)} columns. "
                f"Proceed to Step 2 below.</div>"
            ))

        except Exception as e:
            import traceback                    # P5
            tb = traceback.format_exc()
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
                f"color:#721c24; margin-bottom:10px;'>❌ <b>Error loading data:</b> {e}</div>"
            ))
            print(tb)
            return

    # called outside the with-block; guarded so df is always bound
    if df is not None:
        _initiate_mapping_interface(df)

# --- LOCAL FILE LOGIC ---
def on_file_upload(change):
    with log_output:
        clear_output(wait=True)
        try:
            dd_local_sheets.layout.display = 'none'
            btn_process_file.disabled = True
            fname, content_bytes = get_uploaded_file_data(uploader)
            if not fname: return

            display(HTML(f"<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-bottom: 10px;'>🔎 <b>Analyzing file structure:</b> '{fname}'...</div>"))

            if fname.endswith(('.xls', '.xlsx')):
                excel_file = pd.ExcelFile(io.BytesIO(content_bytes))
                sheets = excel_file.sheet_names
                dd_local_sheets.options = sheets
                dd_local_sheets.value = sheets[0]
                dd_local_sheets.layout.display = 'block'
                clear_output(wait=True)
                if len(sheets) > 1:
                    display(HTML(f"<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>Excel file recognized.</b> Found {len(sheets)} sheets. Please select the target sheet below.</div>"))
                else:
                    display(HTML("<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>Excel file ready.</b> 1 sheet found.</div>"))
            else:
                clear_output(wait=True)
                display(HTML("<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>CSV file structure verified.</b> Ready to load.</div>"))
            btn_process_file.disabled = False
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-bottom: 10px;'>❌ <b>Error reading file structure:</b> {e}</div>"))

def on_process_file_clicked(b):
    with log_output:
        clear_output(wait=True)
        df = None                               # P1: always initialise df
        try:
            fname, content_bytes = get_uploaded_file_data(uploader)

            # P4: explicit None / empty guard before any use of content_bytes
            if not fname or content_bytes is None:
                display(HTML(
                    "<div style='padding:10px; background-color:#fff3cd; border-left:4px solid #ffc107; "
                    "color:#856404; margin-bottom:10px;'>"
                    "⚠️ <b>No file detected.</b> Please select a file before clicking Load.</div>"
                ))
                return

            if fname.endswith('.csv'):
                display(HTML(
                    "<div style='padding:10px; background-color:#e2e3e5; border-left:4px solid #6c757d; "
                    "color:#383d41; margin-bottom:10px;'>📥 <b>Parsing CSV data...</b></div>"
                ))
                df = _safe_read_csv(content_bytes)

            else:
                sheet_target = dd_local_sheets.value
                display(HTML(
                    f"<div style='padding:10px; background-color:#e2e3e5; border-left:4px solid #6c757d; "
                    f"color:#383d41; margin-bottom:10px;'>📥 <b>Parsing Excel sheet:</b> '{sheet_target}'...</div>"
                ))
                df = pd.read_excel(io.BytesIO(content_bytes), sheet_name=sheet_target)
                if df.columns.duplicated().any():
                    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
                    display(HTML(
                        f"<div style='padding:10px; background-color:#fff3cd; border-left:4px solid #ffc107; "
                        f"color:#856404; margin-bottom:10px;'>"
                        f"⚠️ <b>Duplicate column names detected:</b> {dupes}. "
                        f"Pandas has auto-renamed them. Consider fixing the source file.</div>"
                    ))

            df.columns = [str(c).strip() for c in df.columns]

            # P6 — empty DataFrame guard
            if df.empty:
                display(HTML(
                    "<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
                    "color:#721c24; margin-bottom:10px;'>"
                    "❌ <b>The file contains no data rows.</b> "
                    "Please check that the file has at least one row below the header.</div>"
                ))
                return

            global temp_raw_df
            temp_raw_df = df
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:10px; background-color:#d4edda; border-left:4px solid #28a745; "
                f"color:#155724; margin-bottom:10px;'>✅ <b>Data successfully loaded!</b> "
                f"Analyzed {len(df)} rows and {len(df.columns)} columns. "
                f"Proceed to Step 2 below.</div>"
            ))

        except Exception as e:
            clear_output(wait=True)
            import traceback                    # P5: preserve traceback
            tb = traceback.format_exc()
            display(HTML(
                f"<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
                f"color:#721c24; margin-bottom:10px;'>❌ <b>File processing error:</b> {e}</div>"
            ))
            print(tb)                           # visible in cell output / logs
            return

    # P1: only call if df was successfully assigned
    if df is not None:
        _initiate_mapping_interface(df)


# =============================================================================
# LOGIC PART 2: COLUMN MAPPING (The "Bridge")
# =============================================================================
def _initiate_mapping_interface(df):
    global _data_type_widget

    with step2_header_output:
        clear_output(wait=True)

        display(HTML("""
        <h3 style='color:#2E86AB; margin-bottom:10px; border-bottom: 2px solid #3498db; padding-bottom: 5px;'>Step 2: Select Data Type & Map Columns</h3>
        <div style='background-color:#e7f2fa; padding:15px; border-radius:6px; color:#2c3e50; margin-bottom:15px; border-left: 4px solid #3498db;'>
            <b>Please select your input data format:</b><br>
            <ul style='margin-top: 8px; margin-bottom: 0;'>
                <li><b>Raw Data:</b> You have means, standard deviations, and sample sizes for your Treatment and Control groups.</li>
                <li><b>Pre-calculated:</b> You already have standardized effect sizes (e.g., Hedges' g) and variances calculated.</li>
            </ul>
        </div>
        """))

        _data_type_widget = widgets.RadioButtons(
            options=[
                ('Raw Data - Continuous (Means/SDs/N)', 'raw_continuous'),
                ('Raw Data - Binary (Events/Non-Events)', 'raw_binary'),
                ('Pre-calculated (Effect/SE)', 'pre_calculated')
            ],
            value='raw_continuous',
            description='Data Type:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='600px')
        )
        display(_data_type_widget)

        def _on_data_type_change(change):
            _render_column_mapping(df, change['new'])

        _data_type_widget.observe(_on_data_type_change, names='value')

    # Trigger the first render outside the with block to ensure reliable layout rendering
    _render_column_mapping(df, _data_type_widget.value)


def _render_column_mapping(df, data_type):
    with step2_body_output:
        clear_output(wait=True)
        if data_type == 'raw_continuous':
            _render_raw_mapping(df)
        elif data_type == 'raw_binary':
            _render_binary_mapping(df)
        else:
            _render_precalc_mapping(df)

def _render_binary_mapping(df):
    """Column mapping for binary raw data (Events/Non-Events) — includes geographic section."""
    FIELD_INFO = {
        'id':          {'label': 'Study ID / Label:', 'desc': 'Unique name for the study or paper (e.g., "Smith 2020").'},
        'events_e':    {'label': 'Treatment Events (a):', 'desc': 'Number of events (successes/cases) in the Treatment group.'},
        'nonevents_e': {'label': 'Treatment Non-Events (b):', 'desc': 'Number of non-events (failures/controls) in the Treatment group.'},
        'events_c':    {'label': 'Control Events (c):', 'desc': 'Number of events (successes/cases) in the Control group.'},
        'nonevents_c': {'label': 'Control Non-Events (d):', 'desc': 'Number of non-events (failures/controls) in the Control group.'}
    }
    cols_lower = [str(c).lower().strip() for c in df.columns]
    mapping_widgets = {}
    ui_rows = [widgets.HTML("<hr><h4 style='color:#2E86AB; margin-top:15px;'>Map Your Columns (Binary Data Mode)</h4>")]

    for std_name, synonyms in BINARY_COLUMN_SPECS.items():
        guessed_val = None
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        w = widgets.Dropdown(
            options=list(df.columns),
            value=guessed_val,
            description=FIELD_INFO[std_name]['label'],
            style={'description_width': '220px'},
            layout=widgets.Layout(width='600px')
        )
        mapping_widgets[std_name] = w
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:225px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{FIELD_INFO[std_name]['desc']}</i></div>")
        ]))

    # --- Geographic section (optional) ---
    geo_vbox, geo_widgets = _build_geo_mapping_widgets(df)
    ui_rows.append(geo_vbox)

    # Finalize button
    btn_finalize = widgets.Button(
        description="✓ Confirm Mapping & Finalize Data",
        button_style='success',
        layout=widgets.Layout(width='600px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )
    finalize_output = widgets.Output()

    def on_finalize_clicked(b):
        with finalize_output:
            clear_output(wait=True)
            try:
                col_map = {k: w.value for k, w in mapping_widgets.items()}
                if None in col_map.values():
                    missing = [k for k, v in col_map.items() if v is None]
                    raise ValueError(f"Please select a column for: {', '.join(missing)}")
                if len(set(col_map.values())) != len(col_map.values()):
                    raise ValueError("Duplicate mapping detected. You cannot map one column to two fields.")

                global raw_data_standardized, DATA_TYPE_SELECTED, GEO_COLUMNS_MAPPED
                mapped_cols = list(col_map.values())

                geo_col_originals = [w.value for w in geo_widgets.values() if w.value != 'None']
                all_reserved = set(mapped_cols + geo_col_originals)
                extra_cols = [c for c in df.columns if c not in all_reserved]

                raw_data_standardized = df[mapped_cols + geo_col_originals + extra_cols].copy()
                raw_data_standardized.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

                geo_map, geo_warn_html, geo_summ_html, geo_type = _finalize_geo_columns(
                    raw_data_standardized, geo_widgets, mapped_cols
                )
                GEO_COLUMNS_MAPPED = geo_map

                # CRITICAL: We set this to 'raw' so downstream cells treat it as raw data that needs processing
                DATA_TYPE_SELECTED = 'raw'

                summary_html = _build_moderator_summary_html(
                    raw_data_standardized, mapped_cols, extra_cols, "Raw Binary Mode",
                    geo_warnings_html=geo_warn_html,
                    geo_summary_html=geo_summ_html,
                    geo_type=geo_type
                )
                display(HTML(summary_html))
                mark_stale(ALL_DOWNSTREAM, "Step 2: Binary Data Mapping Changed")
            except Exception as e:
                _show_finalize_error(e)

    btn_finalize.on_click(on_finalize_clicked)
    ui_rows.append(btn_finalize)
    ui_rows.append(finalize_output)
    display(widgets.VBox(ui_rows, layout=widgets.Layout(padding='10px')))

def _render_raw_mapping(df):
    """Column mapping for raw data mode — now includes geographic section."""
    FIELD_INFO = {
        'id':  {'label': 'Study ID / Label:', 'desc': 'Unique name for the study or paper (e.g., "Smith 2020").'},
        'xe':  {'label': 'Experimental Mean (xe):', 'desc': 'Mean outcome for the Treatment group.'},
        'sde': {'label': 'Experimental SD (sde):', 'desc': 'Standard Deviation for the Treatment group.'},
        'ne':  {'label': 'Experimental N (ne):', 'desc': 'Sample size for the Treatment group.'},
        'xc':  {'label': 'Control Mean (xc):', 'desc': 'Mean outcome for the Control group.'},
        'sdc': {'label': 'Control SD (sdc):', 'desc': 'Standard Deviation for the Control group.'},
        'nc':  {'label': 'Control N (nc):', 'desc': 'Sample size for the Control group.'}
    }
    cols_lower = [str(c).lower().strip() for c in df.columns]
    mapping_widgets = {}
    ui_rows = [widgets.HTML("<hr><h4 style='color:#2E86AB; margin-top:15px;'>Map Your Columns (Raw Data Mode)</h4>")]

    for std_name, synonyms in RAW_COLUMN_SPECS.items():
        guessed_val = None
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        w = widgets.Dropdown(
            options=list(df.columns),
            value=guessed_val,
            description=FIELD_INFO[std_name]['label'],
            style={'description_width': '180px'},
            layout=widgets.Layout(width='600px')
        )
        mapping_widgets[std_name] = w
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:185px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{FIELD_INFO[std_name]['desc']}</i></div>")
        ]))

    # --- Geographic section (optional) ---
    geo_vbox, geo_widgets = _build_geo_mapping_widgets(df)
    ui_rows.append(geo_vbox)

    # Finalize button
    btn_finalize = widgets.Button(
        description="✓ Confirm Mapping & Finalize Data",
        button_style='success',
        layout=widgets.Layout(width='600px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )
    finalize_output = widgets.Output()

    def on_finalize_clicked(b):
        with finalize_output:
            clear_output(wait=True)
            try:
                col_map = {k: w.value for k, w in mapping_widgets.items()}
                if None in col_map.values():
                    missing = [k for k, v in col_map.items() if v is None]
                    raise ValueError(f"Please select a column for: {', '.join(missing)}")
                if len(set(col_map.values())) != len(col_map.values()):
                    raise ValueError("Duplicate mapping detected. You cannot map one column to two fields.")

                global raw_data_standardized, DATA_TYPE_SELECTED, GEO_COLUMNS_MAPPED
                mapped_cols = list(col_map.values())

                geo_col_originals = [w.value for w in geo_widgets.values() if w.value != 'None']
                all_reserved = set(mapped_cols + geo_col_originals)
                extra_cols = [c for c in df.columns if c not in all_reserved]

                raw_data_standardized = df[mapped_cols + geo_col_originals + extra_cols].copy()
                raw_data_standardized.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

                geo_map, geo_warn_html, geo_summ_html, geo_type = _finalize_geo_columns(
                    raw_data_standardized, geo_widgets, mapped_cols
                )
                GEO_COLUMNS_MAPPED = geo_map
                DATA_TYPE_SELECTED = 'raw'

                summary_html = _build_moderator_summary_html(
                    raw_data_standardized, mapped_cols, extra_cols, "Raw Mode",
                    geo_warnings_html=geo_warn_html,
                    geo_summary_html=geo_summ_html,
                    geo_type=geo_type
                )
                display(HTML(summary_html))
                mark_stale(ALL_DOWNSTREAM, "Step 2: Raw Data Mapping Changed")
            except Exception as e:
                _show_finalize_error(e)

    btn_finalize.on_click(on_finalize_clicked)
    ui_rows.append(btn_finalize)
    ui_rows.append(finalize_output)
    display(widgets.VBox(ui_rows, layout=widgets.Layout(padding='10px')))


def _render_precalc_mapping(df):
    """Column mapping for pre-calculated effect sizes — now includes geographic section."""
    FIELD_INFO = {
        'id':       {'label': 'Study ID / Label:', 'desc': 'Unique identifier for each study.', 'required': True},
        'yi':       {'label': 'Effect Size (yi):', 'desc': "The calculated effect size (e.g., Hedges' g, lnRR, etc.).", 'required': True},
        'variance': {'label': 'Variance (vi):', 'desc': 'The variance of the effect size.', 'required': False},
        'se':       {'label': 'Standard Error (SE):', 'desc': 'The standard error (will convert to variance if needed).', 'required': False},
        'n_total':  {'label': 'Sample Size (n_total):', 'desc': 'Total sample size (optional — useful for diagnostics).', 'required': False}
    }
    cols_lower = [str(c).lower().strip() for c in df.columns]
    mapping_widgets = {}
    ui_rows = [widgets.HTML("""
    <hr>
    <h4 style='color:#2E86AB; margin-top:15px;'>Map Your Columns (Pre-calculated Mode)</h4>
    <div style='background-color:#e3f2fd; padding:10px; border-radius:5px; margin-bottom:10px;'>
        <b>💡 About Pre-calculated Effect Sizes:</b><br>
        You'll need:<br>
        • <b>Effect Size (yi):</b> The standardized effect (g, lnRR, etc.)<br>
        • <b>Variance OR Standard Error:</b> Map at least one<br>
        • <b>Sample Size (optional):</b> Helps with some diagnostics
    </div>
    """)]

    field_order = ['id', 'yi', 'variance', 'se', 'n_total']
    for std_name in field_order:
        synonyms = PRECALC_COLUMN_SPECS[std_name]
        info = FIELD_INFO[std_name]
        guessed_val = None
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        options = ['None'] + list(df.columns) if not info['required'] else list(df.columns)
        w = widgets.Dropdown(
            options=options,
            value=guessed_val if guessed_val is not None else ('None' if not info['required'] else None),
            description=info['label'],
            style={'description_width': '180px'},
            layout=widgets.Layout(width='600px')
        )
        mapping_widgets[std_name] = w
        req_text = " <b style='color:#c0392b;'>(Required)</b>" if info['required'] else " <i>(Optional)</i>"
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:185px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{info['desc']}</i>{req_text}</div>")
        ]))

    geo_vbox, geo_widgets = _build_geo_mapping_widgets(df)
    ui_rows.append(geo_vbox)

    btn_finalize = widgets.Button(
        description="✓ Confirm Mapping & Finalize Data",
        button_style='success',
        layout=widgets.Layout(width='600px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )
    finalize_output = widgets.Output()

    def on_finalize_clicked(b):
        with finalize_output:
            clear_output(wait=True)
            try:
                col_map = {k: w.value for k, w in mapping_widgets.items()}
                col_map = {k: v for k, v in col_map.items() if v != 'None'}
                if 'id' not in col_map or 'yi' not in col_map:
                    raise ValueError("Please map required fields: Study ID and Effect Size (yi)")
                if 'variance' not in col_map and 'se' not in col_map:
                    raise ValueError("Please map either Variance (vi) OR Standard Error (SE)")
                mapped_values = list(col_map.values())
                if len(set(mapped_values)) != len(mapped_values):
                    raise ValueError("Duplicate mapping detected. You cannot map one column to two fields.")

                global raw_data_standardized, DATA_TYPE_SELECTED, VARIANCE_TYPE_SELECTED, GEO_COLUMNS_MAPPED
                mapped_cols = list(col_map.values())

                geo_col_originals = [w.value for w in geo_widgets.values() if w.value != 'None']
                all_reserved = set(mapped_cols + geo_col_originals)
                extra_cols = [c for c in df.columns if c not in all_reserved]

                raw_data_standardized = df[mapped_cols + geo_col_originals + extra_cols].copy()
                raw_data_standardized.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

                DATA_TYPE_SELECTED = 'pre_calculated'
                if 'variance' in col_map and 'se' in col_map:
                    VARIANCE_TYPE_SELECTED = 'both'
                elif 'variance' in col_map:
                    VARIANCE_TYPE_SELECTED = 'variance'
                else:
                    VARIANCE_TYPE_SELECTED = 'se'

                geo_map, geo_warn_html, geo_summ_html, geo_type = _finalize_geo_columns(
                    raw_data_standardized, geo_widgets, mapped_cols
                )
                GEO_COLUMNS_MAPPED = geo_map

                extra_info = (
                    f"<br>Effect Size Column: <b>{col_map['yi']}</b> &nbsp;·&nbsp; "
                    f"Variance Type: <b>{VARIANCE_TYPE_SELECTED.upper()}</b>"
                )
                summary_html = _build_moderator_summary_html(
                    raw_data_standardized, mapped_cols, extra_cols, "Pre-calculated Mode",
                    extra_info,
                    geo_warnings_html=geo_warn_html,
                    geo_summary_html=geo_summ_html,
                    geo_type=geo_type
                )
                display(HTML(summary_html))
                mark_stale(ALL_DOWNSTREAM, "Step 2: Pre-calculated Data Mapping Changed")
            except Exception as e:
                _show_finalize_error(e)

    btn_finalize.on_click(on_finalize_clicked)
    ui_rows.append(btn_finalize)
    ui_rows.append(finalize_output)
    display(widgets.VBox(ui_rows, layout=widgets.Layout(padding='10px')))

# =============================================================================
# REPRODUCIBILITY WIDGETS (TAB 3)
# =============================================================================
_repro_upload = widgets.FileUpload(
    accept='.json',
    multiple=False,
    description='Load Settings JSON',
    layout=widgets.Layout(width='300px')
)
_repro_output = widgets.Output()

def _on_repro_upload(change):
    with _repro_output:
        clear_output()
        uploaded = change['new']
        if not uploaded:
            return

        file_info = uploaded[0] if isinstance(uploaded, (list, tuple)) else list(uploaded.values())[0]
        content = file_info['content'] if isinstance(file_info, dict) else file_info

        # Ensure safely decoded if newer ipywidgets version returns memoryview
        if hasattr(content, 'tobytes'):
            content = content.tobytes()

        try:
            global ANALYSIS_CONFIG
            ANALYSIS_CONFIG = load_reproducibility_config(content)

            meta = ANALYSIS_CONFIG.get('_reproducibility', {})
            df = ANALYSIS_CONFIG.get('clean_dataframe')

            # --- 1. Data Provenance ---
            exp_date = meta.get('exported_at', 'Unknown')
            if 'T' in exp_date:
                exp_date = exp_date.split('.')[0].replace('T', ' ')
            data_shape = f"<b>{len(df)}</b> rows, <b>{len(df.columns)}</b> columns" if df is not None else "Unknown"
            d_type = ANALYSIS_CONFIG.get('data_type', 'raw').upper()

            # --- 2. Pre-processing Details ---
            pre_col = ANALYSIS_CONFIG.get('prefilter_col', 'None')
            pre_vals = ANALYSIS_CONFIG.get('prefilter_values', [])
            if pre_col != 'None' and pre_vals:
                filter_html = f"Filtered on <code>{pre_col}</code> (Kept {len(pre_vals)} categories)"
            else:
                filter_html = "<i style='color:#6c757d;'>None applied</i>"

            sd_miss = ANALYSIS_CONFIG.get('sd_missing_strategy', 'None').replace('_', ' ').title()
            sd_zero = ANALYSIS_CONFIG.get('sd_zero_strategy', 'None').replace('_', ' ').title()
            cv_val = ANALYSIS_CONFIG.get('custom_cv', '')
            cv_str = f" ({cv_val})" if 'Custom' in sd_miss and cv_val else ""
            sd_html = f"Missing: <b>{sd_miss}{cv_str}</b> &nbsp;|&nbsp; Zero: <b>{sd_zero}</b>" if d_type == 'RAW' else "<i style='color:#6c757d;'>N/A (Pre-calculated)</i>"

            # --- 3. Modeling Details ---
            es_type = ANALYSIS_CONFIG.get('effect_size_type', 'Not selected')

            gs = ANALYSIS_CONFIG.get('global_settings', {})
            if gs:
                gs_html = f"<b>{gs.get('model_choice', 'Auto')}</b> &nbsp;|&nbsp; "
                gs_html += f"τ²: <b>{gs.get('tau_method', 'REML')}</b> &nbsp;|&nbsp; "
                gs_html += f"<b>{gs.get('dist_type', 't').upper()}</b>-dist &nbsp;|&nbsp; "
                gs_html += f"α=<b>{gs.get('alpha', 0.05)}</b>"
            else:
                gs_html = "<i style='color:#6c757d;'>Default settings</i>"

            sub_config = ANALYSIS_CONFIG.get('subgroup_config', {})
            if sub_config:
                sub_html = f"<b>{sub_config.get('analysis_type', 'Unknown').title().replace('_', ' ')}</b> by "
                sub_html += f"<code>{sub_config.get('moderator1', 'None')}</code>"
                if sub_config.get('moderator2'):
                    sub_html += f" × <code>{sub_config.get('moderator2')}</code>"
                sub_html += f" <span style='color:#6c757d; font-size: 11px;'>(Min k={sub_config.get('min_obs')}, Studies={sub_config.get('min_papers')})</span>"
            else:
                sub_html = "<i style='color:#6c757d;'>None saved</i>"

            # --- Generate HTML Card ---
            display(HTML(f"""
            <div style='font-family: sans-serif; max-width: 850px;'>
                <!-- Success Banner -->
                <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; color: #155724; border-radius: 4px; margin-bottom: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                    <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>✅</span> Reproducibility Session Loaded Successfully</h4>
                    <p style='margin: 0; font-size: 14px;'>Historical data and settings have been fully hydrated into memory.</p>
                </div>

                <!-- Session Details -->
                <div style='background-color: #ffffff; border: 1px solid #dee2e6; border-radius: 4px; margin-bottom: 15px; overflow: hidden; box-shadow: 0 2px 4px rgba(0,0,0,0.02);'>
                    <h5 style='margin: 0; color: #2c3e50; background-color: #f8f9fa; padding: 12px 15px; border-bottom: 1px solid #dee2e6;'>Session Configuration Overview</h5>

                    <div style='padding: 15px;'>
                        <table style='width: 100%; font-size: 13.5px; color: #333; border-collapse: collapse;'>

                            <!-- Data Section -->
                            <tr><td colspan="2" style='color: #6c757d; font-size: 11px; text-transform: uppercase; font-weight: bold; padding: 10px 0 4px 0; border-bottom: 1px solid #eee;'>1. Data Provenance</td></tr>
                            <tr><td style='padding: 6px 0; width: 160px; color: #495057;'>Export Date:</td><td>{exp_date}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Data Hydrated:</td><td>{data_shape}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Input Format:</td><td><span style='background: #e9ecef; padding: 2px 6px; border-radius: 3px; font-size: 12px;'>{d_type}</span></td></tr>

                            <!-- Pre-processing Section -->
                            <tr><td colspan="2" style='color: #6c757d; font-size: 11px; text-transform: uppercase; font-weight: bold; padding: 15px 0 4px 0; border-bottom: 1px solid #eee;'>2. Pre-Processing</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Global Filters:</td><td>{filter_html}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Variance Imputation:</td><td>{sd_html}</td></tr>

                            <!-- Modeling Section -->
                            <tr><td colspan="2" style='color: #6c757d; font-size: 11px; text-transform: uppercase; font-weight: bold; padding: 15px 0 4px 0; border-bottom: 1px solid #eee;'>3. Modeling Settings</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Effect Size Metric:</td><td><span style='color: #0056b3; font-weight: bold;'>{es_type}</span></td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Overall Analysis:</td><td>{gs_html}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Subgroup Config:</td><td>{sub_html}</td></tr>

                        </table>
                    </div>
                </div>

                <!-- Next Steps -->
                <div style='padding: 15px; background-color: #e8f4f8; border-left: 5px solid #17a2b8; color: #0c5460; border-radius: 4px;'>
                    <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>🚀</span> Next Step</h4>
                    <p style='margin: 0; font-size: 14px;'>Do not click any other buttons in this step. <b>Simply run the subsequent cells in order</b> (Step 3, Step 4, etc.). The interface will automatically restore the settings above.</p>
                </div>
            </div>
            """))


        except Exception as e:
            print(f"ERROR loading config: {e}")
            import traceback
            traceback.print_exc()

_repro_upload.observe(_on_repro_upload, names='value')

# --- Tab 3: Load Settings JSON (Reproducibility) ---
repro_tab_content = widgets.VBox([
    widgets.HTML("""
    <div style='background:#e8f4f8; border-left:5px solid #17a2b8; padding:15px;
         border-radius:5px; margin:10px 0;'>
      <h4 style='margin-top:0; color:#0c5460;'>Restore Previous Session</h4>
      <p style='color:#0c5460; margin-bottom:5px;'>
        Upload a previously exported <code>analysis_settings.json</code> file.
        Once loaded, <b>run the remaining notebook cells sequentially</b>.
        The pipeline will automatically reconstruct the data and select the exact settings used previously.
      </p>
    </div>
    """),
    _repro_upload,
    _repro_output
], layout=widgets.Layout(padding='10px'))

# =============================================================================
# INITIALIZE UI
# =============================================================================
btn_auth.on_click(on_auth_clicked)
btn_fetch_ws.on_click(on_fetch_ws_clicked)
btn_load_sheet.on_click(on_load_sheet_clicked)
uploader.observe(on_file_upload, names='value')
btn_process_file.on_click(on_process_file_clicked)

# --- Tab 3: Load Settings JSON (Reproducibility) ---
repro_tab_content = widgets.VBox([
    widgets.HTML("""
    <div style='background:#e8f4f8; border-left:5px solid #17a2b8; padding:15px;
         border-radius:5px; margin:10px 0;'>
      <h4 style='margin-top:0; color:#0c5460;'>Reproducibility Mode</h4>
      <p style='color:#0c5460; margin-bottom:5px;'>
        Upload a previously exported <code>analysis_settings.json</code> file to
        instantly restore all data and perfectly replicate a previous session.
      </p>
    </div>
    """),
    widgets.HBox([_repro_upload]),
    _repro_output
], layout=widgets.Layout(padding='10px'))

# --- Tab 4: Built-in Examples ---
dd_example_data = widgets.Dropdown(
    options=list(BUILT_IN_DATASETS.keys()),
    description='Select Data:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

btn_load_example = widgets.Button(
    description="📥 Load Example Dataset",
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='300px', height='40px')
)

example_tab_content = widgets.VBox([
    widgets.HTML("""
    <div style='background:#e8f4f8; border-left:5px solid #17a2b8; padding:15px; border-radius:5px; margin:10px 0;'>
      <h4 style='margin-top:0; color:#0c5460;'>Built-in Example Datasets</h4>
      <p style='color:#0c5460; margin-bottom:5px;'>
        Load classic meta-analysis datasets instantly to test the pipeline. These are pre-mapped
        and require no manual configuration.
      </p>
    </div>
    """),
    dd_example_data,
    btn_load_example
], layout=widgets.Layout(padding='10px'))

tabs = widgets.Tab(children=[gs_vbox, local_vbox, repro_tab_content, example_tab_content])
tabs.set_title(0, "Google Sheets")
tabs.set_title(1, "Upload Excel/CSV")
tabs.set_title(2, "Restore Session")
tabs.set_title(3, "Built-in Examples")

# --- Example Data Logic ---
def on_load_example_clicked(b):
    with step2_header_output:
        clear_output()
        selection = dd_example_data.value
        config = BUILT_IN_DATASETS[selection]

        # 1. Build DataFrame from hardcoded dictionary
        df_example = pd.DataFrame(config['data'])
        data_type_raw = config['type']  # e.g., 'raw_continuous' or 'raw_binary'

        # 2. Sanitize the data type for the backend pipeline!
        # If it starts with 'raw', the backend just needs to see 'raw'.
        pipeline_data_type = 'raw' if data_type_raw.startswith('raw') else 'pre_calculated'

        # 3. Inject directly into global standardized variables
        global raw_data_standardized, DATA_TYPE_SELECTED, VARIANCE_TYPE_SELECTED
        raw_data_standardized = df_example.copy()
        DATA_TYPE_SELECTED = pipeline_data_type

        if pipeline_data_type == 'pre_calculated':
            VARIANCE_TYPE_SELECTED = 'variance'

        # 4. Clear previous ANALYSIS_CONFIG to prevent stale data overlap
        global ANALYSIS_CONFIG
        ANALYSIS_CONFIG = {
            'data_type': pipeline_data_type,
            'clean_dataframe': df_example.copy(),
            'prefilter_col': 'None',
            'prefilter_values': []
        }

        # 5. Create a success message bypassing the manual mapper
        display(HTML(f"""
        <div style='background-color:#d4edda; color:#155724; padding:15px; border-radius:8px; border:1px solid #c3e6cb; margin-bottom:15px;'>
            <h4 style='margin:0 0 8px 0;'>✅ Example Data Loaded Successfully</h4>
            <b>{len(df_example)}</b> rows loaded &nbsp;·&nbsp;
            <b>Mode:</b> {data_type_raw.upper().replace('_', ' ')}<br><br>
            <i>The data columns are already pre-mapped to the backend. You can skip the mapping step below and proceed directly to <b>Step 3 (Global Filtering)</b> or <b>Step 4 (Data Cleaning)</b>!</i>
        </div>
        """))

        # Clear the body output so they don't see the manual mapping UI
        with step2_body_output:
            clear_output()

btn_load_example.on_click(on_load_example_clicked)


display(HTML("<h3 style='color:#2E86AB; border-bottom: 2px solid #3498db; padding-bottom: 5px;'>Step 1: Import Data Source</h3>"))
display(tabs)
display(log_output)
display(step2_header_output)
display(step2_body_output)